In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 2. Load Dataset
# =========================
path=r"C:\machine learning\repository\Dataset for regression.csv"

df=pd.read_csv(path)
df.head()
# =========================
# 3. Convert Date Columns
# =========================
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], errors='coerce')

# =========================
# 4. Create Target Variable
# =========================
df['delivery_time'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()

# Remove rows where target is missing
df = df.dropna(subset=['delivery_time'])

# =========================
# 5. Feature Engineering
# =========================
df['order_hour'] = df['created_at'].dt.hour
df['order_day'] = df['created_at'].dt.dayofweek

df['is_weekend'] = df['order_day'].isin([5, 6]).astype(int)

# Avoid division issue using safe formula
df['busy_ratio'] = df['total_busy_dashers'] / (df['total_onshift_dashers'] + 1)

# =========================
# 6. Drop Unnecessary Columns
# =========================
df = df.drop(['created_at', 'actual_delivery_time'], axis=1)

# =========================
# 7. Handle Missing + Infinite Values
# =========================

df = pd.get_dummies(df, drop_first=True)

# Then fix infinity & NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.fillna(0)

# Now apply clip (only numeric data exists)
df = df.clip(-1e6, 1e6)

#
# =========================
# 9. Define Features and Target
# =========================
X = df.drop('delivery_time', axis=1)
y = df['delivery_time']

# =========================
# 10. Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 11. Train Model
# =========================
model = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 12. Prediction
# =========================
y_pred = model.predict(X_test)

# =========================
# 13. Evaluation
# =========================
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

C:\Users\perfect\AppData\Local\Temp\ipykernel_2068\3696908638.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
C:\Users\perfect\AppData\Local\Temp\ipykernel_2068\3696908638.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], errors='coerce')


MAE: 2322.442661916883
RMSE: 9172.366974594494
R2 Score: 0.6267982920685861
